# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring. **Question:** which pages should a reviewer look at first?
**Data:** the same starter slice as my Week-4 baseline (`data/raw/content_refresh_anonymized.csv`, 30,000 pages, 32 clients).
**Label:** `is_declining = (trend_direction == "down")` — same as Week 4.
**Metric:** precision@K (of the top K pages, how many are declining), next to the base rate — same as Week 4.
**Seed:** 42 everywhere.

In [1]:
import os, sys, json, subprocess, warnings
import numpy as np, pandas as pd, sklearn
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/susheel123-sketch/Flyrank-Internship-ml", "Flyrank-Internship-ml"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int).values
groups = df["client_id"].values
print("rows:", len(df), "| clients:", df["client_id"].nunique(), f"| base rate: {y.mean():.3f}")
print("sklearn", sklearn.__version__, "| pandas", pd.__version__)

rows: 30000 | clients: 32 | base rate: 0.542
sklearn 1.9.1 | pandas 3.0.5


## 1. Method choice and why

**My lane is a ranking question:** a reviewer works down a list, so what matters is the *order*, not a yes/no per page.
So I train classifiers on the yes/no label and **rank by their probability**, then grade at precision@K — exactly how the Week-4 rule was graded.

| Model | Why it is on the list |
|---|---|
| **Logistic Regression** | The readable one. Each feature gets one number I can explain. If it ties the bigger models, I keep it. |
| **Decision Tree (depth 3)** | Printable. Shows the 3–4 splits the data cares about most. Not expected to win. |
| **Random Forest** | The standard "stronger" step. Handles odd shapes (e.g. very big pages behaving differently). |
| **Gradient Boosting** | Usually the strongest on tables like this. "Safe" here because the split is grouped and the features are leak-checked. |

**Not used:** clustering (my question has a label, so I don't need to invent groups); the warehouse (blocked from where this ran — same slice as Week 4 keeps the comparison fair).

**Features:** the same no-leak rule as Week 4 — no `trend_*` columns and no last-30 / prev-30 windows. Counts are logged
(traffic is heavy-tailed). Missing values get a `has_` flag instead of a silent fill, because missingness follows content type.

In [2]:
# Features: same no-leak rule as Week 4 — nothing from the label or the two 30-day windows that build it.
FORBIDDEN = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
LOG_COLS = ["impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d", "ai_sessions_90d",
            "search_volume", "cpc"]
NUM_COLS = LOG_COLS + ["competition", "word_count", "days_with_impressions", "days_with_sessions",
                       "content_age_days", "days_since_last_update", "ctr", "avg_position",
                       "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CAT_COLS = ["competition_level", "content_type", "main_intent"]
WINDOW_SHAPE = ["days_with_impressions", "days_with_sessions"]   # tested separately below (ablation)

def build_features(d, drop=()):
    X = pd.DataFrame(index=d.index)
    for c in NUM_COLS:
        if c in drop:
            continue
        v = np.log1p(d[c]) if c in LOG_COLS else d[c]
        X[c] = v.fillna(0)
        if d[c].isna().any():                     # missingness follows content_type -> flag it, don't hide it
            X["has_" + c] = d[c].notna().astype(int)
    X["has_position"] = (d["avg_position"] > 0).astype(int)   # avg_position 0 = "no data"
    X = X.join(pd.get_dummies(d[CAT_COLS].fillna("unknown"), dtype=int))
    return X

X = build_features(df)
assert not set(X.columns) & set(FORBIDDEN), "label-derived column in features"
assert "client_id" not in X.columns and "content_id" not in X.columns
print(X.shape[1], "features:", list(X.columns))

MODELS = {
    "Logistic Regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "Decision Tree (depth 3)": lambda: DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED),
    "Gradient Boosting": lambda: HistGradientBoostingClassifier(random_state=SEED),
}

36 features: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'search_volume', 'has_search_volume', 'cpc', 'has_cpc', 'competition', 'has_competition', 'word_count', 'has_word_count', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'has_scroll_rate', 'ai_traffic_pct', 'has_position', 'competition_level_HIGH', 'competition_level_LOW', 'competition_level_MEDIUM', 'competition_level_unknown', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown']


## 2. Split design

**GroupKFold by client, 5 folds.** Every page is tested by a model that has never seen *any* page from its client.

- **Why grouped:** pages from one client share a site, a writer, a niche and often one batch-update date (Week 4 showed this).
  A random split would let the model memorise "this client declines" and look better than it is on a new client.
- **Why not time-aware:** this slice is one snapshot (one 90-day window), so there is no "earlier vs later" to split on.
  That is a real limit: every result here is a **same-window link, not a forecast**. A forward-window test needs the warehouse.
- **Fairness to the baseline:** the Week-4 rule had 3 numbers learned from data (median CTR per position band).
  Inside each fold I re-learn them from the **training clients only**, so the baseline gets no peek at test clients either.
- **Watch-out shown below:** fold 0 is a single large client (7,008 pages), and fold base rates swing from 0.38 to 0.65.
  That is why I report the spread across folds, not just one average.

In [3]:
gkf = GroupKFold(n_splits=5)
FOLDS = list(gkf.split(X, y, groups))
fold_info = []
for i, (tr, te) in enumerate(FOLDS):
    assert not set(groups[tr]) & set(groups[te]), "client leaked across the split"
    fold_info.append({"fold": i, "test_pages": len(te), "test_clients": len(set(groups[te])),
                      "test_base_rate": round(y[te].mean(), 3),
                      "biggest_client_share_of_test": round(pd.Series(groups[te]).value_counts(normalize=True).iloc[0], 2)})
display(pd.DataFrame(fold_info))
print("No client appears in both train and test in any fold (asserted).")

def baseline_score(train, test):
    # Week-4 rule, unchanged — except the 3 band-CTR medians are learned from the TRAIN clients only.
    band = lambda d: pd.cut(d["avg_position"].where(d["avg_position"] > 0), [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
    btr = band(train); ok = (train["impressions_90d"] >= 500) & btr.notna()
    med = train[ok].groupby(btr[ok], observed=True)["ctr"].median()
    bte = band(test); exp_ctr = bte.map(med).astype(float)
    elig = (test["impressions_90d"] >= 500) & bte.notna()
    flag = elig & (test["ctr"] < 0.5 * exp_ctr)
    gap = (1 - test["ctr"] / exp_ctr).clip(0, 1)
    score = np.where(flag, gap * np.log1p(test["impressions_90d"]), 0.0)
    return score + np.log1p(test["impressions_90d"].values) * 1e-6     # same tie-break as Week 4 (impressions)

def p_at_k(y_true, score, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return float(np.asarray(y_true)[order[:k]].mean())

,fold,test_pages,test_clients,test_base_rate,biggest_client_share_of_test
0,0,7008,1,0.490,1.00
1,1,5731,7,0.645,0.64
2,2,5753,8,0.379,0.40
3,3,5755,8,0.622,0.39
4,4,5753,8,0.585,0.31


No client appears in both train and test in any fold (asserted).


## 3. Train + compare vs my baseline

Two views, both on the same split and metric:
1. **Per fold** — rank each held-out client group on its own, then average over the 5 folds (with spread).
2. **Pooled** — one queue over all 30k pages using the out-of-fold scores. This is the view that matches my Week-4 table.

In [4]:
KS = [20, 50, 100]
oof = {name: np.zeros(len(df)) for name in ["Week-4 baseline"] + list(MODELS)}
fold_rows = []
for i, (tr, te) in enumerate(FOLDS):
    s = baseline_score(df.iloc[tr], df.iloc[te]); oof["Week-4 baseline"][te] = s
    fold_rows.append({"fold": i, "model": "Week-4 baseline", **{f"P@{k}": p_at_k(y[te], s, k) for k in KS},
                      "AUC": np.nan, "base_rate": y[te].mean()})
    for name, make in MODELS.items():
        m = make().fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]; oof[name][te] = p
        fold_rows.append({"fold": i, "model": name, **{f"P@{k}": p_at_k(y[te], p, k) for k in KS},
                          "AUC": roc_auc_score(y[te], p), "base_rate": y[te].mean()})
folds_df = pd.DataFrame(fold_rows)

order = ["Week-4 baseline"] + list(MODELS)
mean = folds_df.groupby("model")[[f"P@{k}" for k in KS] + ["AUC"]].mean().loc[order]
std = folds_df.groupby("model")[[f"P@{k}" for k in KS]].std().loc[order]
summary = mean.round(3).astype(str)
for c in std.columns:
    summary[c] = mean[c].map("{:.3f}".format) + " ± " + std[c].map("{:.3f}".format)
base_w = folds_df.query("model=='Week-4 baseline'").set_index("fold")
summary["folds beating baseline @P50"] = [
    "—" if n == "Week-4 baseline" else
    f"{int((folds_df.query('model==@n').set_index('fold')['P@50'] > base_w['P@50']).sum())} of 5" for n in order]
print(f"Per-client-fold results (mean ± std over 5 held-out client folds). Base rate per fold ≈ {y.mean():.3f}")
display(summary)

Per-client-fold results (mean ± std over 5 held-out client folds). Base rate per fold ≈ 0.542


,P@20,P@50,P@100,AUC,folds beating baseline @P50
model,,,,,
Week-4 baseline,0.740 ± 0.096,0.760 ± 0.113,0.766 ± 0.121,NaN,—
Logistic Regression,0.820 ± 0.076,0.776 ± 0.105,0.802 ± 0.101,0.659,2 of 5
Decision Tree (depth 3),0.650 ± 0.050,0.620 ± 0.091,0.600 ± 0.073,0.611,0 of 5
Random Forest,0.790 ± 0.129,0.824 ± 0.091,0.782 ± 0.072,0.679,3 of 5
Gradient Boosting,0.850 ± 0.061,0.832 ± 0.044,0.798 ± 0.072,0.685,3 of 5


In [5]:
# Same view as Week 4: one pooled queue over all 30k pages, each page scored by a model that never saw its client.
POOL_KS = [10, 20, 50, 100, 200, 500]
pooled = pd.DataFrame({name: [p_at_k(y, s, k) for k in POOL_KS] for name, s in oof.items()},
                      index=[f"P@{k}" for k in POOL_KS]).T.loc[order].round(3)
pooled["base_rate"] = round(y.mean(), 3)
print("Pooled out-of-fold queue (comparable to the Week-4 table: baseline P@10 0.60, P@50 0.64, P@100 0.69)")
display(pooled)

Pooled out-of-fold queue (comparable to the Week-4 table: baseline P@10 0.60, P@50 0.64, P@100 0.69)


,P@10,P@20,P@50,P@100,P@200,P@500,base_rate
Week-4 baseline,0.6,0.65,0.60,0.66,0.685,0.688,0.542
Logistic Regression,0.7,0.80,0.86,0.81,0.820,0.788,0.542
Decision Tree (depth 3),0.5,0.60,0.60,0.57,0.600,0.600,0.542
Random Forest,0.5,0.70,0.72,0.75,0.780,0.786,0.542
Gradient Boosting,0.9,0.90,0.84,0.84,0.845,0.810,0.542


**Reading the two tables honestly:**

- **Pooled (the Week-4 view): the models win clearly.** Gradient Boosting P@50 = 0.84 vs baseline 0.60; Logistic Regression 0.86.
  The baseline's top-10 weakness from Week 4 is exactly where the models gain most.
- **Per fold: the win is smaller and noisy.** Baseline P@50 = 0.76 ± 0.11; Gradient Boosting 0.83 ± 0.04 and beats the baseline in 3 of 5 folds;
  Logistic Regression only 2 of 5. With ~6 clients per fold, one client can swing a fold by 10 points.
- **Why the two views differ:** inside one group of clients, the rule's flag is a decent sorter. Across many clients, the rule's
  scores are not on a common scale, while a model's probabilities are. So most of the model's extra value is **better ordering across clients**.
- **The depth-3 tree loses to the baseline everywhere.** Useful only to read (section 4).
- **Ablation:** dropping the two window-shape features changes Logistic Regression by almost nothing and costs Gradient Boosting
  ~1–5 points. So the win does not ride on them.

In [6]:
# Ablation: drop the two 'window-shape' features (how many of the 90 days had traffic).
# They share the label's window most directly, so I want to know how much of the win rides on them.
X_strict = build_features(df, drop=WINDOW_SHAPE)
abl = []
for name in ["Logistic Regression", "Gradient Boosting"]:
    for label, XX in [("all features", X), ("without window-shape", X_strict)]:
        vals = []
        for tr, te in FOLDS:
            p = MODELS[name]().fit(XX.iloc[tr], y[tr]).predict_proba(XX.iloc[te])[:, 1]
            vals.append([p_at_k(y[te], p, k) for k in KS] + [roc_auc_score(y[te], p)])
        abl.append({"model": name, "features": label, **dict(zip([f"P@{k}" for k in KS] + ["AUC"], np.mean(vals, axis=0).round(3)))})
abl_df = pd.DataFrame(abl); display(abl_df)

metrics = {"assignment": "ML-08", "data": "starter slice", "split": "GroupKFold(5) by client_id", "seed": SEED,
           "sklearn": sklearn.__version__, "base_rate": round(float(y.mean()), 4),
           "per_fold_mean": mean.round(4).to_dict(orient="index"),
           "per_fold_std": std.round(4).to_dict(orient="index"),
           "pooled_oof": pooled.drop(columns="base_rate").to_dict(orient="index"),
           "ablation_no_window_shape": abl, "features": list(X.columns)}
os.makedirs("work/outputs", exist_ok=True)
json.dump(metrics, open("work/outputs/w05_model_metrics.json", "w"), indent=2, default=float)
print("wrote work/outputs/w05_model_metrics.json")

,model,features,P@20,P@50,P@100,AUC
0,Logistic Regression,all features,0.82,0.776,0.802,0.659
1,Logistic Regression,without window-shape,0.82,0.792,0.812,0.661
2,Gradient Boosting,all features,0.85,0.832,0.798,0.685
3,Gradient Boosting,without window-shape,0.80,0.820,0.796,0.659


wrote work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

**Which model I pick: Gradient Boosting** — and only after checking it earns the extra complexity:
- best per-fold P@50 with the **smallest spread** (0.83 ± 0.04), top pooled P@10/P@20;
- Logistic Regression scores close on average, but its top 50 turns out to be mostly **zero-click pages** (shown in 4e) —
  one pattern that may be a tracking problem, not a page problem. Gradient Boosting's list is more varied.
- Random Forest is close behind but less stable at the top. Logistic Regression stays as my readable cross-check.

In [7]:
# 4a. What does the simple model lean on? Logistic Regression coefficients (features are standardised,
# so sizes are comparable: + pushes toward 'declining', − away from it). Fit on all rows only to READ it.
lr = MODELS["Logistic Regression"]().fit(X, y)
coef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
display(coef.head(10).round(3).to_frame("LR coefficient (standardised)"))

# 4b. Does that hold when checked by shuffling? Permutation importance on HELD-OUT clients (fold 0), drop in AUC.
tr, te = FOLDS[0]
for name in ["Logistic Regression", "Gradient Boosting"]:
    m = MODELS[name]().fit(X.iloc[tr], y[tr])
    pi = permutation_importance(m, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
    imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
    print(f"{name}: top-6 permutation importance (AUC drop when shuffled, held-out fold 0)")
    display(imp.head(6).round(4).to_frame("AUC drop"))

# 4c. The depth-3 tree, printed — the most readable version of what the data says.
tree = MODELS["Decision Tree (depth 3)"]().fit(X, y)
print(export_text(tree, feature_names=list(X.columns), decimals=1))

,LR coefficient (standardised)
impressions_90d,0.920
has_position,0.867
clicks_90d,-0.676
avg_position,-0.287
content_age_days,-0.231
search_volume,-0.162
has_word_count,0.139
sessions_90d,-0.123
days_since_last_update,0.111
competition,0.109


Logistic Regression: top-6 permutation importance (AUC drop when shuffled, held-out fold 0)


,AUC drop
clicks_90d,0.0566
impressions_90d,0.0508
avg_position,0.0489
content_age_days,0.0362
search_volume,0.0095
competition_level_HIGH,0.0049


Gradient Boosting: top-6 permutation importance (AUC drop when shuffled, held-out fold 0)


,AUC drop
avg_position,0.0805
impressions_90d,0.0336
content_age_days,0.0293
days_with_impressions,0.0167
clicks_90d,0.0062
scroll_rate,0.0030


|--- days_with_impressions <= 4.5
|   |--- avg_position <= 0.8
|   |   |--- ai_traffic_pct <= 150.0
|   |   |   |--- class: 0
|   |   |--- ai_traffic_pct >  150.0
|   |   |   |--- class: 0
|   |--- avg_position >  0.8
|   |   |--- content_age_days <= 107.5
|   |   |   |--- class: 0
|   |   |--- content_age_days >  107.5
|   |   |   |--- class: 0
|--- days_with_impressions >  4.5
|   |--- content_age_days <= 344.5
|   |   |--- clicks_90d <= 3.3
|   |   |   |--- class: 1
|   |   |--- clicks_90d >  3.3
|   |   |   |--- class: 0
|   |--- content_age_days >  344.5
|   |   |--- avg_position <= 21.3
|   |   |   |--- class: 1
|   |   |--- avg_position >  21.3
|   |   |   |--- class: 0



In [8]:
# 4d. Where is the chosen model wrong? Use the out-of-fold scores (every page scored by a model that never saw its client).
PICK = "Gradient Boosting"
e = df[["content_id", "client_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
        "days_with_impressions", "content_age_days", "content_type", "main_intent"]].copy()
e["y"] = y; e["score"] = oof[PICK]
e["traffic"] = pd.cut(e["impressions_90d"], [0, 100, 1000, 10000, 1e9], labels=["<100", "100-1k", "1k-10k", "10k+"])

# Per-client: how good is the top of each client's own queue? (a reviewer works one client at a time)
pc = (e.groupby("client_id")
        .apply(lambda g: pd.Series({"pages": len(g), "base_rate": g["y"].mean(),
                                    "model_P@20": g.nlargest(20, "score")["y"].mean()}))
        .query("pages >= 200"))
pc["lift"] = pc["model_P@20"] - pc["base_rate"]
print(f"Per-client top-20 (clients with >=200 pages, n={len(pc)}): model beats the client's own base rate in "
      f"{(pc['lift'] > 0).sum()} of {len(pc)}")
display(pc.sort_values("lift").round(3).head(5).reset_index(drop=True).rename_axis("worst clients (pseudonyms hidden)"))

# By traffic tier: score vs reality
tier = e.groupby("traffic", observed=True).agg(n=("y", "size"), actual_decline=("y", "mean"), mean_score=("score", "mean")).round(3)
print("By traffic tier — does the score track reality?"); display(tier)

# Top-50 pooled false positives: what do they look like vs the true positives?
top50 = e.nlargest(50, "score")
print(f"Pooled top 50: {top50['y'].sum()} declining, {(1 - top50['y']).sum()} not")
display(top50.groupby("y")[["impressions_90d", "ctr", "avg_position", "days_with_impressions", "content_age_days"]].median()
        .rename(index={0: "false positives (median)", 1: "true positives (median)"}).round(2))

# Three concrete wrong cases: highest-scored pages that did NOT decline
wrong = top50[top50["y"] == 0].head(3)[["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
                                         "days_with_impressions", "content_age_days", "content_type", "score"]]
display(wrong.round(3))

Per-client top-20 (clients with >=200 pages, n=22): model beats the client's own base rate in 19 of 22


,pages,base_rate,model_P@20,lift
worst clients (pseudonyms hidden),,,,
0,649.0,0.194,0.10,-0.094
1,556.0,0.595,0.55,-0.045
2,476.0,0.000,0.00,0.000
3,325.0,0.197,0.25,0.053
4,1043.0,0.937,1.00,0.063


By traffic tier — does the score track reality?


,n,actual_decline,mean_score
traffic,,,
<100,8006,0.389,0.423
100-1k,8485,0.603,0.625
1k-10k,9907,0.620,0.632
10k+,3602,0.524,0.518


Pooled top 50: 42 declining, 8 not


,impressions_90d,ctr,avg_position,days_with_impressions,content_age_days
y,,,,,
false positives (median),864.0,0.01,4.9,67.5,202.0
true positives (median),1583.5,0.00,3.6,81.0,148.0


,content_id,impressions_90d,clicks_90d,ctr,avg_position,days_with_impressions,content_age_days,content_type,score
26544,content_3bb23afaf34e,2910,1,0.03,15.4,88,144,keyword article,0.958
23206,content_f854023b075b,627,0,0.00,5.1,59,258,keyword article,0.948
25112,content_2e27654f487a,975,1,0.10,2.8,73,257,keyword article,0.942


In [9]:
# 4e. Two checks the error table pointed at.
# (1) Zero-click pages: visible (>=1k impressions), has a position, but 0 clicks in 90 days.
zc = (df["clicks_90d"] == 0) & (df["impressions_90d"] >= 1000) & (df["avg_position"] > 0)
print(f"Zero-click visible pages: {zc.sum()} ({zc.mean():.1%} of all) | their decline rate: {y[zc].mean():.3f} vs {y[~zc].mean():.3f} for the rest")
share = pd.DataFrame({name: [zc.values[np.argsort(-s, kind='stable')[:k]].mean() for k in [50, 100, 500]]
                      for name, s in oof.items()}, index=["top 50", "top 100", "top 500"]).T.loc[order].round(2)
print("Share of each queue that is zero-click pages:"); display(share)
# (2) How much of the label is a client-level event rather than a page-level one?
cb = pd.Series(y).groupby(df["client_id"].values).agg(["size", "mean"]).query("size >= 200")["mean"]
print(f"Client base rates (clients with >=200 pages, n={len(cb)}): min {cb.min():.2f}, median {cb.median():.2f}, max {cb.max():.2f}")
print(f"Clients with a base rate below 0.2 or above 0.8: {((cb < 0.2) | (cb > 0.8)).sum()} of {len(cb)}")

Zero-click visible pages: 1149 (3.8% of all) | their decline rate: 0.667 vs 0.537 for the rest


Share of each queue that is zero-click pages:


,top 50,top 100,top 500
Week-4 baseline,0.16,0.21,0.48
Logistic Regression,0.72,0.62,0.44
Decision Tree (depth 3),0.04,0.05,0.05
Random Forest,0.18,0.13,0.15
Gradient Boosting,0.20,0.23,0.14


Client base rates (clients with >=200 pages, n=22): min 0.00, median 0.55, max 0.94
Clients with a base rate below 0.2 or above 0.8: 6 of 22


**What the errors look like:**

- **What the models lean on is sensible, not suspicious.** No single feature dominates (top permutation drops are 0.03–0.08 AUC — nothing near "perfect").
  The top three for Gradient Boosting:
  1. **Position** — pages on page 1–2 have the most visibility to lose.
  2. **Impressions** — busy-but-not-huge pages decline most; the very biggest are steadier (the same volume pattern as Week 4).
  3. **Content age** — younger pages decline more (the first-months bump fading). The tree shows the same thing.
  Logistic Regression's biggest weights are impressions (+) and clicks (−) — that is the **low-CTR signal from Week 4**, learned instead of hand-written.
- **Pages with almost no visibility are the easy "no".** The tree's first split: fewer than ~5 days with impressions → not declining.
  Pages that barely show up have nothing to lose.
- **False positives in the top 50 are smaller, patchier and older** than the true positives
  (median 864 vs 1,584 impressions; 68 vs 81 days with impressions; 202 vs 148 days old). Small, patchy pages make noisy trends.
- **Three concrete misses (top-scored, did not decline):**
  1. `content_3bb23afaf34e` — 2,910 impressions, 1 click, position 15.4. Low CTR on page 2 that simply held steady. Low CTR predicts decline; it does not guarantee it.
  2. `content_f854023b075b` — 627 impressions, 0 clicks, visible on only 59 of 90 days. Too little traffic for a stable trend; it could have gone either way.
  3. `content_2e27654f487a` — 975 impressions, 1 click, position **2.8**. Top-3 with ~0% CTR again — likely the results page answers the search itself. The model cannot see that.
- **Client is the biggest error source.** Client decline rates run from 0.00 to 0.94, and 6 of 22 sizeable clients sit below 0.2 or above 0.8.
  For those, "declining" looks like a **site-wide event**, not a page problem — no page-level feature can rank inside them. The model still beats each client's own base rate in 19 of 22 clients.
- **Zero-click pages are a trap for the simple model.** 1,149 visible pages have 0 clicks; they do decline more (0.67 vs 0.54), but Logistic Regression fills 72% of its top 50 with them.
  If those zeros are tracking gaps (Week-4 review suspected this), that queue would send reviewers to fix titles on pages that are fine.

In [10]:
# The final table: model vs baseline, same split, same metric, same run.
final = pooled[["P@10", "P@20", "P@50", "P@100"]].copy()
final.columns = ["pooled P@10", "pooled P@20", "pooled P@50", "pooled P@100"]
final["per-fold P@50 (mean ± std)"] = summary["P@50"]
final["folds beating baseline @P50"] = summary["folds beating baseline @P50"]
final["AUC (per-fold mean)"] = mean["AUC"].round(3)
final["base rate"] = round(y.mean(), 3)
display(final)

,pooled P@10,pooled P@20,pooled P@50,pooled P@100,per-fold P@50 (mean ± std),folds beating baseline @P50,AUC (per-fold mean),base rate
Week-4 baseline,0.6,0.65,0.60,0.66,0.760 ± 0.113,—,NaN,0.542
Logistic Regression,0.7,0.80,0.86,0.81,0.776 ± 0.105,2 of 5,0.659,0.542
Decision Tree (depth 3),0.5,0.60,0.60,0.57,0.620 ± 0.091,0 of 5,0.611,0.542
Random Forest,0.5,0.70,0.72,0.75,0.824 ± 0.091,3 of 5,0.679,0.542
Gradient Boosting,0.9,0.90,0.84,0.84,0.832 ± 0.044,3 of 5,0.685,0.542


## Conclusion

- **Gradient Boosting beats my Week-4 rule** on the same pages, same client-grouped split and same metric: pooled P@50 0.84 vs 0.60, per-fold P@50 0.83 vs 0.76.
- **The win is real but smaller than the pooled number suggests.** Per fold it wins 3 of 5, and the folds are noisy. Most of the gain is better ordering across clients.
- **Simpler is almost as good.** Logistic Regression is close on average. I picked boosting for its stability and its less one-sided top list — not because it is complex.
- **Everything is a same-window, observed link.** It is decision-support for "who to review first", not a forecast of future decline. The next honest test is a forward window on the warehouse.
- **Reproducible:** seed 42, scikit-learn 1.9.1. Tree models can move a few points between library versions.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed before commit; seed 42)
- [x] No client names, URLs, or private queries anywhere (pseudonymous IDs only; worst-client table hides them)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Baseline and models compared on the same split, same metric, same run
- [x] Grouped-by-client validation; no label-derived or 30-day-window inputs (asserted)
- [x] Simpler model preferred where the complex one does not clearly earn its keep
- [x] Committed under `work/notebooks/`; receipts in `work/outputs/w05_model_metrics.json`